# 第3章 AIエージェント 最小構成

LLM自身に「次に呼ぶツール(関数)」を選ばせる **エージェント** の最小例を体験します。

**所要時間**: 約20分

> 💡 **モデルに関する補足**
> 本ハンズオンでは Amazon Nova Lite を使用します。Nova シリーズは `ChatBedrockConverse` 経由で Tool Calling に対応していますが、`langchain-aws` のバージョンや Nova モデル自体の挙動の違いにより、ツール呼び出しがうまく動かないケースが報告されたこともあります。
> もしこの章で `executor.invoke(...)` がエラーになったり、ツールが呼ばれず最終回答が不自然な場合は、`.env` の `BEDROCK_CHAT_MODEL_ID` を以下に変更して試してください:
>
> ```
> BEDROCK_CHAT_MODEL_ID=anthropic.claude-3-haiku-20240307-v1:0
> ```
>
> ⚠️ Anthropic 製モデルは自動有効化されますが、**初回利用時に一度だけ「使用フォーム」(use case詳細)の提出が必要** です(数分で承認されます)。詳細は [docs/00_setup.md](../docs/00_setup.md#anthropic製モデルを使う場合オプション) を参照。

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

# .env を読み込む(これまでと同じ)
load_dotenv()

# 第3章では LLM のみ使用(埋め込みは不要)
AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

## 1. ツール(=ただのPython関数)を定義

`@tool` デコレータを付けると、LangChainがLLMに渡せるツールスキーマに変換してくれます。

**`@tool` の働き**
1. 関数のシグネチャ(引数名・型・デフォルト値)を読み取り、JSONスキーマを自動生成
2. 関数の **docstring を「ツールの説明」** として LLM に渡す
3. ツール名は関数名がそのまま使われる

LLM はこの「名前・説明・引数スキーマ」を見て、どのツールを呼ぶべきかを判断します。
docstring が雑だと LLM が誤ったツールを選びがちなので、**docstring は丁寧に書く** のがコツです。

**型ヒントが重要な理由**
- `timezone: str = "Asia/Tokyo"` のような型ヒントは、JSONスキーマの `type: "string"` 等に変換される
- LLM は「文字列を渡せばいいんだな」と理解し、正しい型で呼び出してくれる

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
# @tool は普通の関数を「LLMに渡せるツール」に変換するデコレータ
from langchain_core.tools import tool


# @tool デコレータでツール化。関数本体は普通の Python 関数のまま動く
@tool
def get_current_time(timezone: str = "Asia/Tokyo") -> str:
    """指定したタイムゾーンの現在時刻を ISO 8601 文字列で返す。
    timezone は IANA 形式(例: 'Asia/Tokyo', 'UTC')。"""
    # ↑ この docstring が LLM への "ツール説明" として渡される
    return datetime.now(ZoneInfo(timezone)).isoformat(timespec="seconds")


@tool
def add_numbers(a: float, b: float) -> float:
    """2つの数値を加算した結果を返す。"""
    return a + b


# LLM に渡すために、ツールをリストにまとめる
tools = [get_current_time, add_numbers]

# 各ツールに自動付与された name と description を確認
# (name = 関数名、 description = docstring)
for t in tools:
    print(f"- {t.name}: {t.description}")

## 2. エージェントを構築

- `create_tool_calling_agent`: Tool Calling 対応モデル向けの Agent ファクトリ
- `AgentExecutor`: エージェントを実行するランナー(LLM → Tool → LLM のループを回す)

**Agent と AgentExecutor の役割分担**

| 役割 | 担当 |
|---|---|
| 「次に何を呼ぶか」決める | **Agent**(プロンプト + LLM + 解析の塊) |
| 実際にツール関数を呼ぶ → 結果を Agent に戻す → 終了判定する | **AgentExecutor** |

つまり Agent 単体は「次のアクション」を返すだけで、それを **実行してループを回すのが AgentExecutor** という分担です。

**`MessagesPlaceholder("agent_scratchpad")` とは**
ループの途中で、過去のツール呼び出しと結果のやり取り(= scratchpad)を蓄積する場所です。
LLM はこの履歴を見て「次は別のツールを呼ぶ?」「もう十分?最終回答?」を判断します。
おまじない的ですが、Tool Calling Agent では必ずこのプレースホルダを入れます。

**`verbose=True` の効果**
AgentExecutor の各ステップ(LLMの思考、ツール呼び出し、結果)を標準出力に表示します。
デバッグや学習にとても役立つので、本ハンズオンでは ON にしてあります。

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# Agent作成用ファクトリと、Agentを実際にループ実行させるExecutor
from langchain.agents import AgentExecutor, create_tool_calling_agent

# LLM 準備(これまでと同じ)
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# プロンプト定義。ポイントは MessagesPlaceholder("agent_scratchpad")
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "あなたはツールを使いこなすアシスタントです。\n"
     "必要に応じてツールを呼び、その結果を踏まえて日本語で簡潔に答えてください。"),
    ("human", "{input}"),  # ユーザーの質問が入る
    # agent_scratchpad: 過去のツール呼び出しと結果の履歴が、ループのたびに自動でここに挿入される
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Agent を作成。内部では「LLMに tools をbindして、決まった形式で出力させる」設定が行われる
agent = create_tool_calling_agent(llm, tools, prompt)

# Executor を作成。 verbose=True で各ステップが標準出力に表示される
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 3. 実行: 時刻を尋ねる

LLMが `get_current_time` を呼ぶはずです。`verbose=True` のログに注目してください。

**verbose ログの読み方**
```
> Entering new AgentExecutor chain...
Invoking: `get_current_time` with `{'timezone': 'Asia/Tokyo'}`   ← LLMが選んだツールと引数
2026-05-27T...                                                   ← 関数の戻り値
[最終回答]                                                        ← LLMが結果をまとめて返した答え
> Finished chain.
```
ツール選択 → 実行 → 結果を踏まえた回答、という流れが追えます。

In [ ]:
# Executor の入力は {"input": "..."} 形式の dict(プロンプトの {input} に対応)
# 戻り値は {"input": ..., "output": "最終回答"} の dict
result = executor.invoke({"input": "いま東京は何時ですか?"})
print("\n=== 最終回答 ===")
print(result["output"])

## 4. 実行: 計算を頼む

今度は `add_numbers` が選ばれるはずです。

In [ ]:
# 計算が必要な質問 → add_numbers が選ばれるはず
# (LLMは数値計算が苦手なので、Toolに任せた方が正確という典型的な例)
result = executor.invoke({"input": "123.4 と 56.78 を足すといくつ?"})
print("\n=== 最終回答 ===")
print(result["output"])

## 5. 実行: ツールが不要な質問

ツールを呼ばずに LLM が直接答えるパターンも観察します。

In [ ]:
# 一般知識で答えられる質問 → LLMはツールを呼ばず、自分の知識で直接回答する
# verbose ログに "Invoking: ..." 行が現れないことを確認
result = executor.invoke({"input": "フランスの首都はどこ?"})
print("\n=== 最終回答 ===")
print(result["output"])

## まとめ

- `@tool` で Python 関数をエージェントが使えるツールに変換できる
- `create_tool_calling_agent` + `AgentExecutor` で **LLM ↔ Tool のループ** が回る
- LLMは質問に応じて、ツールを呼ぶか / 自分で答えるかを判断する

次は [04 まとめ](../04_wrapup/) で、ここから先の学習リンクを紹介します。